# Expected Wait

## Reporting Grain

Per Stop, Service Pattern, Day Part (not yet implemented), and Reporting Period (currently ad hoc for a week).

## Source Data

GTFS-Schedule and Trip Updates

In [ ]:
import os
from sqlalchemy import create_engine
from jinja2 import Environment, FileSystemLoader
import pandas as pd

from src.data_loading import (
    get_matching_shapes_feeds,
    get_rt_headways,
    get_schedule_feed_entries,
)

### Parameters

In [ ]:
TARGET_DATES = [
    "2026-06-15","2026-06-16","2026-06-17","2026-06-18","2026-06-19"
]
TARGET_ROUTE_SHORT_NAME = os.environ.get("TARGET_ROUTE_SHORT_NAME", "ECR")
TARGET_SCHEDULE_FEED = os.environ.get("TARGET_SCHEDULE_FEED", "Bay Area 511 SamTrans Schedule")
CONSTANT_MACROS = {
    "TARGET_DATES": TARGET_DATES,
    "TARGET_ROUTE_SHORT_NAME": TARGET_ROUTE_SHORT_NAME,
    "TARGET_SCHEDULE_FEED": TARGET_SCHEDULE_FEED
}
CONSTANT_MACROS

In [ ]:
# min count of trips per service period to be considered reliable
MIN_TRIPS = 10

### Implementation

In [ ]:
# gcp boilerplate
CALITP_BQ_MAX_BYTES = os.environ.get("CALITP_BQ_MAX_BYTES", 5_000_000_000)
CALITP_BQ_LOCATION = os.environ.get("CALITP_BQ_LOCATION", "us-west2")
engine = create_engine(
    f"bigquery://cal-itp-data-infra/?maximum_bytes_billed={CALITP_BQ_MAX_BYTES}",  # noqa: E231
    location=CALITP_BQ_LOCATION
)
env = Environment(loader=FileSystemLoader("templates"))


In [ ]:
# Get the target route's service patterns, their geometry, and the feeds behind them
matching_shapes, feed_dates, feed_keys = get_matching_shapes_feeds(
    engine, env, CONSTANT_MACROS, min_trips=MIN_TRIPS
)
display(feed_dates)
matching_shapes

In [ ]:
# Get the schedule feed entry for every feed that was active during the window
feed_dates = get_schedule_feed_entries(engine, env, feed_dates)
feed_dates

In [ ]:
# Get the realtime headway between every pair of consecutive trips, at every
# stop on the pattern rather than only the timepoints
rt_headways = get_rt_headways(
    engine,
    env,
    CONSTANT_MACROS,
    feed_dates=feed_dates,
    shape_ids=matching_shapes.shape_id.to_list(),
)
rt_headways

In [ ]:
# The query carries the headway itself, so the distribution is ready to look at
rt_headways["headway_seconds"].describe()

### Results

In [ ]:
# Headways per stop, pooled over every selected date
headway_summary = (
    rt_headways
    .groupby(["shape_id", "stop_sequence", "stop_id"])
    .agg(
        n_headways=("headway_seconds", "size"),
        mean_headway_seconds=("headway_seconds", "mean"),
        median_headway_seconds=("headway_seconds", "median"),
        max_headway_seconds=("headway_seconds", "max"),
        headway_variance=("headway_seconds", "var"),
    )
    .reset_index()
)
# a passenger arriving at random waits half the mean headway only when service is
# even; the variance term is the penalty for irregular service
headway_summary["expected_wait_seconds"] = (
    headway_summary["mean_headway_seconds"] / 2
    * (
        1
        + headway_summary["headway_variance"]
        / headway_summary["mean_headway_seconds"] ** 2
    )
)
headway_summary["excess_wait_seconds"] = (
    headway_summary["expected_wait_seconds"]
    - headway_summary["mean_headway_seconds"] / 2
)
headway_summary

In [ ]:
# Total headways observed per day
# note fewer headways on start and end of the week
rt_headways["first_actual_departure"].dt.date.value_counts()

In [ ]:
# The longest gaps in service, which are what riders actually notice
(
    rt_headways
    .sort_values("headway_seconds", ascending=False)
    .head(20)
    [["shape_id", "stop_sequence", "stop_id", "first_actual_departure", "second_actual_arrival", "headway_seconds"]]
)